In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, SpectralClustering, AgglomerativeClustering, DBSCAN, Birch
import warnings
warnings.filterwarnings("ignore")

In [2]:
file_path = './FCA_lung_0.01.h5ad'
adata = sc.read_h5ad(file_path)
sc.pp.subsample(adata, fraction=0.01)  
print(adata)
adata.obs['cell_type']
num_unique_cell_types = len(adata.obs['cell_type'].cat.categories)
num_unique_cell_types

AnnData object with n_obs × n_vars = 726 × 58521
    obs: 'cell_type', 'batch'
    var: 'n_cells'


9

In [3]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
adata = adata[:, adata.var['highly_variable']]
sc.tl.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [4]:
sc.tl.leiden(adata, key_added='leiden')
print("Leiden labels:", adata.obs['leiden'].unique())
kmeans = KMeans(n_clusters=8, random_state=18)  
kmeans_labels = kmeans.fit_predict(adata.obsm['X_pca'])  
adata.obs['kmeans'] = kmeans_labels.astype(str)
print("KMeans labels:", adata.obs['kmeans'].unique())
spectral = SpectralClustering(n_clusters=8, random_state=18, affinity='nearest_neighbors')
spectral_labels = spectral.fit_predict(adata.obsm['X_pca'])
adata.obs['spectral'] = spectral_labels.astype(str)
print("Spectral labels:", adata.obs['spectral'].unique())

Leiden labels: ['2', '0', '4', '3', '1', ..., '9', '7', '5', '6', '10']
Length: 11
Categories (11, object): ['0', '1', '2', '3', ..., '7', '8', '9', '10']
KMeans labels: ['4' '3' '0' '1' '5' '2' '6' '7']
Spectral labels: ['0' '1' '4' '2' '3' '7' '6' '5']


In [5]:
cell_type_list = adata.obs['cell_type'].tolist()
unique_cell_types = adata.obs['cell_type'].unique().tolist()
print("Unique cell types:", unique_cell_types)
clustering_results = adata.obs[['leiden', 'kmeans', 'spectral']].copy()
print(clustering_results.head())
clustering_results.to_csv('clustering_results0313.csv', index=True)

Unique cell types: ['Stromal cells', 'Bronchiolar and alveolar epithelial cells', 'Lymphoid cells', 'Vascular endothelial cells', 'Ciliated epithelial cells', 'Neuroendocrine cells', 'Myeloid cells', 'Megakaryocytes', 'Lymphatic endothelial cells']
      leiden kmeans spectral
3590       2      4        0
23721      0      3        1
28544      4      0        4
57284      3      1        2
56489      2      4        1


In [6]:
from ghtree import process_clustering_network_part, process_clustering_network_all
process_clustering_network_part(
    csv_path='clustering_results0313.csv',
    leiden_col='leiden',
    target_cluster=2,
    output_ttree_path="Ttree0313part.graphml",
    output_gtree_path="Gtree0313G.graphml"
)

previous: 
       leiden  kmeans  spectral
3590        2       4         0
23721       0       3         1
28544       4       0         4
57284       3       1         2
56489       2       4         1
New node for leiden cluster 2: 0


  0%|                                                                                            | 0/3 [00:00<?, ?it/s]
%|                                                                                           | 0/10 [00:00<?, ?it/s]
%|████████▎                                                                          | 1/10 [00:00<00:00,  9.82it/s]
 33%|████████████████████████████                                                        | 1/3 [00:00<00:00,  4.28it/s]
%|                                                                                            | 0/8 [00:00<?, ?it/s]
%|█████████████████████                                                               | 2/8 [00:00<00:01,  4.47it/s]
 67%|████████████████████████████████████████████████████████                            | 2/3 [00:00<00:00,  2.26it/s]
%|                                                                                            | 0/8 [00:00<?, ?it/s]
100%|██████████████████████████████████████████████████

{'weight': 1} {'weight': 8} {'weight': 2}
G save to 'Gtree0313G.graphml'


100%|██████████████████████████████████████████████████████████████████████| 102644/102644 [00:00<00:00, 526723.57it/s]


Ttree save to 'Ttree0313part.graphml'


In [14]:
from processpart import process_clustering_part
from processall import process_clustering_all
process_clustering_part(
    graphml_path="Ttree0313part.graphml",
    csv_path='clustering_results0313.csv',
    leiden_col='leiden',
    target_cluster=2,
    start_node="0",
    threshold=20,
    new_col_name='leiden_new',
    output_path='updated_clustering_results.csv',
    task=2
)

previous:
       leiden  kmeans  spectral
3590        2       4         0
23721       0       3         1
28544       4       0         4
57284       3       1         2
56489       2       4         1

 cankao
10%: 1.0
20%: 217.0
25%: 226.0
33%: 339.0
50%: 449.0
217.0

Added nodes: {'51928', '32654', '7423', '60616', '16007', '50529', '14917', '48265', '4898', '6914', '22117', '52142', '5878', '24846', '31072', '47182', '26491', '12876', '11708', '31787', '51857', '26762', '714', '19494', '7455', '3485', '45257', '71043', '43767', '13583', '28628', '19768', '6384', '31781', '53717', '40264', '12211', '53566', '15678', '11081', '27058', '12679', '15554', '0', '4461', '51948', '33496', '56679', '35908', '16146', '24020', '38667', '67226', '37859', '72341', '58043', '58417', '52896'}
after:
       leiden  kmeans  spectral  leiden_new
3590        2       4         0           2
23721       0       3         1           0
28544       4       0         4           4
57284       3       1   